## What problem is Opik solving?

Imagine you build a RAG application:

```text
User Question
      ↓
Retriever
      ↓
LLM
      ↓
Answer
```

The user says:

> "The answer is wrong."

Now you need to know:

- Was retrieval bad?
- Was context missing?
- Did the LLM hallucinate?
- Was a tool slow?
- Which prompt was used?
- How many tokens were consumed?

Without observability, you only see:

```python
answer = "Paris"
```

With Opik, you see the entire execution tree:

```text
User Question
      ↓
retrieve()
      ↓
generate_answer()
      ↓
evaluate()
```

Every step becomes visible in the dashboard.

### What Opik Gives You

Instead of only seeing the final answer, you can inspect:

- Inputs and outputs of each component
- Retrieved documents
- Prompt versions
- LLM responses
- Latency for every step
- Token usage and cost
- Evaluation metrics and scores
- Traces across the entire pipeline

This makes debugging RAG systems significantly easier because you can pinpoint **where** the failure occurred rather than simply knowing that the final answer was incorrect.

## Demo 1: Trace a Simple LLM Pipeline

In [1]:
# Install necessary libraries: opik for tracing and openai for LLM integration.
!pip install opik openai -q

In [2]:
from getpass import getpass
import opik

# Configure Opik with your API key and workspace. This connects your local execution
# to the Opik dashboard for visualization and analysis of traces.
opik.configure(
    api_key=getpass("Enter OPIK_API_KEY: "),
    workspace="aniruddha-mukherjee"
)

OPIK: You already have an API key set in the configuration file. If you want to change it, please use the --force flag or force=True when calling the configure() method. Otherwise, the configuration file will not be updated but the session will use the new API key.
OPIK: Opik is already configured. You can check the settings by viewing the config file at /root/.opik.config
OPIK: Configuration completed successfully. Traces will be logged to 'test-aiml' project. To change the destination project, see: https://www.comet.com/docs/opik/tracing/log_traces#configuring-the-project-name


### Create Tracked Functions

In [3]:
from opik import track
import time

# The `@track` decorator automatically instruments the function for Opik.
# It captures inputs, outputs, execution time, and any nested tracked calls.
@track
def retrieve_context(question):
    # Simulate a delay for context retrieval
    time.sleep(1)

    # Return a dummy context based on the question
    return """
    Paris is the capital of France.
    France is located in Europe.
    """

@track
def generate_answer(question, context):
    # Simulate a delay for answer generation
    time.sleep(2)

    # Return a dummy answer
    return "Paris"

@track
def rag_pipeline(question):
    # This function orchestrates the RAG (Retrieval-Augmented Generation) flow.
    # Opik will trace the calls to `retrieve_context` and `generate_answer` as nested steps.
    context = retrieve_context(question)

    answer = generate_answer(
        question,
        context
    )

    return answer

In [4]:
# Execute the RAG pipeline with a sample question.
# Opik will log the entire execution trace to your dashboard.
response = rag_pipeline(
    "What is the capital of France?"
)

# Print the final answer from the pipeline.
print(response)

OPIK: Started logging traces to the "test-aiml" project at https://www.comet.com/opik/api/v1/session/redirect/projects/?trace_id=019e935a-26ca-730f-8d30-8b53a1a48c7a&path=aHR0cHM6Ly93d3cuY29tZXQuY29tL29waWsvYXBpLw==.


Paris


## Demo 2: Simulating an Agent Workflow

This demo shows why traces are useful.

In [5]:
from opik import track
import random

@track
def web_search(query):
    # Simulates a web search operation, returning a mock result.
    return f"Search results for {query}"

@track
def calculator(expression):
    # Simulates a calculator tool, evaluating a given expression.
    return eval(expression)

@track
def agent(question):
    # This function simulates an agent that uses multiple tools.
    # Opik will trace each tool call (`web_search`, `calculator`) as separate steps within the agent's trace.
    search_result = web_search(question)

    calculation = calculator("10*20")

    return {
        "search": search_result,
        "calculation": calculation
    }

In [6]:
# Call the agent function to demonstrate its workflow and tracing in Opik.
# The output will show the results from the simulated web search and calculation.
agent("Revenue of company")

{'search': 'Search results for Revenue of company', 'calculation': 200}

## Demo 3: OpenAI Integration

In [26]:
# This cell is a duplicate of the initial installation cell. Removing its content.
# The libraries are already installed.

In [7]:
from getpass import getpass
from opik.integrations.openai import track_openai
from openai import OpenAI

# `track_openai` wraps the OpenAI client to automatically capture
# all API calls (e.g., chat completions) and log them as traces in Opik.
client = track_openai(
    OpenAI(api_key=getpass("Enter OPENAI_API_KEY: "))
)

In [8]:
# Make a call to the OpenAI API for a chat completion.
# This call will be automatically traced by Opik due to the `track_openai` wrapping.
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role":"user",
            "content":"Explain transformers"
        }
    ]
)

# Print the content of the LLM's response.
print(response.choices[0].message.content)

Transformers are a type of neural network architecture that has become the foundation for many state-of-the-art models in natural language processing (NLP) and beyond. Introduced in the paper "Attention is All You Need" by Vaswani et al. in 2017, transformers offer a novel approach to handling sequential data like text, images, and more. Here are the key components and concepts associated with transformers:

### Key Concepts

1. **Attention Mechanism**: Central to the transformer architecture is the self-attention mechanism. It allows the model to weigh the importance of different words in a sentence when encoding or decoding information. Instead of processing words in order, as done in traditional recurrent neural networks (RNNs), transformers can look at all words at once.

2. **Multi-Head Attention**: This is an extension of the self-attention mechanism that runs several attention mechanisms in parallel (heads). Each head learns different representations, allowing the model to gathe

## Demo 4: LLM-as-a-Judge

In [13]:
from getpass import getpass
import os

# Set the OpenAI API key as an environment variable. This is often a more secure way
# to handle sensitive keys, especially in production or shared environments.
os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY: ")

In [14]:
from opik.evaluation.metrics import Hallucination

# Import the Hallucination metric from Opik's evaluation module.
# This metric helps assess if an LLM's output contains information not present in the provided context.

In [15]:
# Initialize an instance of the Hallucination metric.
# This prepares the metric for scoring LLM outputs.
metric = Hallucination()

In [18]:
# Define a sample question, a generated answer, and the context used.
# This setup allows for evaluating the 'hallucination' aspect of the answer
# by comparing it against the provided context.
question = "What is the capital of France?"

answer = """
The capital of France is Paris.
"""

context = """
Paris is the capital of France.
"""

In [19]:
# Calculate the hallucination score for the given answer, question, and context.
# The metric will compare the 'answer' to the 'context' to identify unsupported claims.
score = metric.score(
    input=question,
    output=answer,
    context=context
)

# Print the resulting score, which includes the hallucination value and a reason for the score.
print(score)

ScoreResult(name='hallucination_metric', value=0.0, reason="['The output states that the capital of France is Paris, which directly matches the CONTEXT.', 'No new information, misattributions, or contradictions are introduced.']", category_name=None, metadata=None, scoring_failed=False)


In [34]:
# This cell is currently empty. It could be used for further analysis or concluding remarks.